In [1]:
import sys

from evosax.algorithms import Open_ES

sys.path.append('/mnt/new_home/ronedr/evolution-strategy-baselines-comparison')

In [2]:
import os

os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [2]:
import jax
from evosax.problems import CNN, TorchVisionProblem as Problem, identity_output_fn
from tqdm import tqdm
from experiment.run_experiments import run_experiment_permutations

In [3]:
num_generations = 200
population_size = 32
seeds = list(range(0, 5))
result_dir = "../experiment_results"
problems_torch_vision = ["MNIST", "FashionMNIST", "CIFAR10", "SVHN"]

In [3]:
import optax

lr_schedule = optax.exponential_decay(
    init_value=0.01,
    transition_steps=num_generations,
    decay_rate=0.1,
)
std_schedule = optax.exponential_decay(
    init_value=0.05,
    transition_steps=num_generations,
    decay_rate=0.1,
)

es_dict = {
    "Open_ES": {    
        "optimizer": optax.adam(learning_rate=lr_schedule),
        "std_schedule": std_schedule,
    },
    "SimpleES": {},
    "LES": {},
    "DES": {},
    "EvoTF_ES": {},
    "PGPE": {},
    "SNES": {},
    "Sep_CMA_ES": {},
    "CMA_ES": {},
}

In [1]:
for task_name in tqdm(problems_torch_vision, desc="Loading Problems .."):
    try:
        problem = Problem(task_name=task_name,
                          network=CNN(
                              num_filters=[8, 16],
                              kernel_sizes=[(5, 5), (5, 5)],
                              strides=[(1, 1), (1, 1)],
                              mlp_layer_sizes=[10],
                              output_fn=identity_output_fn
                          ),
                          batch_size=32)
        print("Successfully loaded:", task_name)
        for es in es_dict:
            try:
                for seed in seeds:
                    key = jax.random.key(seed)
                    run_experiment_permutations(problems=[problem],
                                                es_dict={es: es_dict[es]},
                                                num_generations=num_generations,
                                                population_size=population_size,
                                                seed=seed,
                                                result_dir=result_dir,
                                                run_again_if_exist=False)
            except:
                continue
    except Exception as e:
        print("Failed to load:", task_name, e)
        continue

NameError: name 'tqdm' is not defined

In [ ]:
from evosax.algorithms import SimpleES

Open_ES